In [22]:
# Imports & Setup
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd, numpy as np

# reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [23]:
df_bt = pd.read_csv("files/welsh_back_translation_high_similarity.csv",
                    usecols=["back_translated_welsh", "cefr_level"])\
         .rename(columns={"back_translated_welsh": "text"})

In [24]:
# Add missing columns 
df_bt["title"] = "Back-translated A1/A2 sample"
df_bt["lang"] = "cy"
df_bt["source_name"] = "back_translation_pipeline"
df_bt["format"] = "text"
df_bt["category"] = "general"
df_bt["license"] = "CC-BY-SA" 

In [25]:
df_bt.head()

,text,cefr_level,title,lang,source_name,format,category,license
0,Roedd Ma/ penbwrdd: fy nhad yn awr yn mynd i'r...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
1,Ar y acw: Byddwch yn troi at y dde yma. Ewch o...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
2,Ac: mae prynhawn da. Ar y tywydd yn ofnadwy! Y...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
3,Ble ydych chi'n byw?,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
4,Beth ydych chi'n hoffi?,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA


In [26]:
df_bt["cefr_level"].value_counts()

cefr_level
A1    243
A2    127
Name: count, dtype: int64

In [27]:
# Welsh CEFR dataset from HuggingFace
ds = load_dataset("UniversalCEFR/learn_welsh_cy")["train"]
df_main = ds.to_pandas()

In [28]:
df_main["cefr_level"].value_counts()

cefr_level
A1    764
A2    608
Name: count, dtype: int64

In [29]:
# Load your B2 JSON data
df_b1 = pd.read_json("files/B1_canolradd_de-learnwelsh.json")
df_b1["cefr_level"] = "B1"
df_b1 = df_b1.drop_duplicates(subset="text", keep="first")

In [30]:
df_b1

,title,lang,source_name,format,category,cefr_level,license,text
0,Uned 1 - Adolygu 2,cy,canolradd-de-learnwelsh,Sentence-level,reference,B1,public,Mae fy nhad i'n dod o Dreorci.
1,Uned 1 - Adolygu 3,cy,canolradd-de-learnwelsh,Sentence-level,reference,B1,public,Mae fy mrawd i'n gweithio men garej.
2,Uned 1 - Adolygu 4,cy,canolradd-de-learnwelsh,Sentence-level,reference,B1,public,Dyw fy chwaer i ddim yn gweithio ar hyn o bryd.
3,Uned 1 - Adolygu 5,cy,canolradd-de-learnwelsh,Sentence-level,reference,B1,public,Roedd fy nhad-cu i'n gweithio ar fferm.
4,Uned 1 - Siaradwch - Trafod pwnc - Cymdogion 1,cy,canolradd-de-learnwelsh,Sentence-level,reference,B1,public,Oes cymdogion da gyda chi?
...,...,...,...,...,...,...,...,...
815,Uned 14 - Lluosog - Credo gan Emyr Davies,cy,canolradd-de-learnwelsh,sentence-level,reference,B1,public,"Ar bwy, yn wir, mae’r bai?"
816,Uned 14 - Lluosog - Credo gan Emyr Davies,cy,canolradd-de-learnwelsh,sentence-level,reference,B1,public,"Mae iaith yn mynd yn ieithoedd,"
817,Uned 14 - Lluosog - Credo gan Emyr Davies,cy,canolradd-de-learnwelsh,sentence-level,reference,B1,public,Mae coeden yn troi’n goed...
818,Uned 14 - Lluosog - Credo gan Emyr Davies,cy,canolradd-de-learnwelsh,sentence-level,reference,B1,public,"Yr iaith Gymraeg, dw i’n meddwl,"


In [31]:
df_b1["cefr_level"].value_counts()

cefr_level
B1    802
Name: count, dtype: int64

In [32]:
# Your B2 JSON
df_b2 = pd.read_json("files/b2_welsh.json")
df_b2["cefr_level"] = "B2"

In [33]:
df_b2["cefr_level"].value_counts()

cefr_level
B2    655
Name: count, dtype: int64

In [36]:
# Combine both DataFrames
df_combined = pd.concat([df_main,df_b1,df_b2,df_bt], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset="text", keep="first")

In [37]:
df_combined["cefr_level"].value_counts()

cefr_level
A1    964
B1    802
A2    719
B2    650
Name: count, dtype: int64

In [38]:
# Keep only A1/A2/B2
label2id = {"A1":0, "A2":1, "B1":2,"B2":3}
df_combined = df_combined[df_combined["cefr_level"].isin(label2id.keys())]
df_combined["label_id"] = df_combined["cefr_level"].map(label2id)

print(df_combined["cefr_level"].value_counts())

cefr_level
A1    964
B1    802
A2    719
B2    650
Name: count, dtype: int64


In [39]:
# Train/Dev/Test Split

train_df, test_df = train_test_split(
    df_combined, test_size=0.2,
    stratify=df_combined["label_id"], random_state=SEED
)
train_df, dev_df = train_test_split(
    train_df, test_size=0.2,
    stratify=train_df["label_id"], random_state=SEED
)

In [40]:
# Dataset & Dataloaders

MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 128

class CEFRDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df["text"].tolist()
        self.labels = df["label_id"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),          # [seq_len]
            "attention_mask": enc["attention_mask"].squeeze(0),# [seq_len]
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = CEFRDataset(train_df, tokenizer, MAX_LEN)
dev_ds   = CEFRDataset(dev_df, tokenizer, MAX_LEN)
test_ds  = CEFRDataset(test_df, tokenizer, MAX_LEN)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)
dev_loader   = torch.utils.data.DataLoader(dev_ds, batch_size=64)
test_loader  = torch.utils.data.DataLoader(test_ds, batch_size=64)

In [41]:
# Multi-Prototype Model (CEFR-SP style)

class XLMR_MultiProto(nn.Module):
    def __init__(self, model_name, num_labels=3, num_prototypes=3, lm_layer=-2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.num_labels, self.K, self.lm_layer = num_labels, num_prototypes, lm_layer
        d = self.encoder.config.hidden_size
        # prototypes: [C,K,d]
        self.prototypes = nn.Parameter(torch.randn(num_labels, num_prototypes, d))
        self.tau = nn.Parameter(torch.tensor(10.0))

    def mean_pool(self, hidden, mask):
        m = mask.unsqueeze(-1).float()
        return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)
    
    def forward(self, input_ids, attention_mask, labels=None, class_weights=None):
        out = self.encoder( 
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            output_hidden_states=True 
        ) 
            
        hidden = out.hidden_states[self.lm_layer] # e.g. -2 for penultimate  
        h = self.mean_pool(hidden, attention_mask) 
        h = F.normalize(h, dim=-1)
            
        P = F.normalize(self.prototypes, dim=-1) 
        sims = torch.einsum("bd,ckd->bck", h, P) # [B,C,K] 
        logits = sims.mean(dim=2) * self.tau # mean over prototypes 
        loss = None 
        if labels is not None: 
            loss = F.cross_entropy(logits, labels, weight=class_weights) 
        return {"loss": loss, "logits": logits}

In [42]:
# Model, Optimizer, Class Weights

model = XLMR_MultiProto(MODEL_NAME, num_labels=len(label2id), num_prototypes=3, lm_layer=-2).to(device)

optimizer = torch.optim.AdamW(
    [{"params": model.encoder.parameters(), "lr": 2e-5},
     {"params": [model.prototypes, model.tau], "lr": 1e-3}],
    weight_decay=0.01
)

# class weights
counts = train_df["label_id"].value_counts().to_dict()
freq = np.array([counts.get(i,0) for i in range(len(label2id))], dtype=np.float32)
freq = np.clip(freq, 1, None)
gamma = 0.75
w = (1.0/freq)**gamma
w = w * (len(w)/w.sum())
class_weights = torch.tensor(w, dtype=torch.float32).to(device)
print("Class weights:", class_weights)


Class weights: tensor([0.8441, 1.0520, 0.9694, 1.1344])


In [43]:
# Training Loop

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_y, all_p = 0.0, [], []
    for batch in loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        y    = batch["labels"].to(device)

        out = model(ids, mask, labels=y, class_weights=class_weights)

        if train:
            out["loss"].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); optimizer.zero_grad()

        total_loss += out["loss"].item()
        preds = out["logits"].argmax(dim=1)
        all_y.append(y.detach().cpu())
        all_p.append(preds.detach().cpu())

    return total_loss/len(loader), torch.cat(all_y).numpy(), torch.cat(all_p).numpy()

EPOCHS = 5
for ep in range(1, EPOCHS+1):
    tr_loss, _, _ = run_epoch(train_loader, train=True)
    va_loss, y_true, y_pred = run_epoch(dev_loader, train=False)
    print(f"\nEpoch {ep}: train_loss={tr_loss:.4f}  val_loss={va_loss:.4f}", flush=True)
    print(classification_report(
        y_true, y_pred,
        labels=list(label2id.values()),
        target_names=list(label2id.keys()),
        digits=3    
    ))


Epoch 1: train_loss=1.2776  val_loss=1.0362
              precision    recall  f1-score   support

          A1      0.632     0.623     0.627       154
          A2      0.423     0.261     0.323       115
          B1      0.687     0.698     0.692       129
          B2      0.507     0.721     0.595       104

    accuracy                          0.580       502
   macro avg      0.562     0.576     0.559       502
weighted avg      0.572     0.580     0.568       502


Epoch 2: train_loss=0.9517  val_loss=0.8495
              precision    recall  f1-score   support

          A1      0.616     0.877     0.724       154
          A2      0.667     0.278     0.393       115
          B1      0.946     0.682     0.793       129
          B2      0.528     0.721     0.610       104

    accuracy                          0.657       502
   macro avg      0.689     0.640     0.630       502
weighted avg      0.694     0.657     0.642       502


Epoch 3: train_loss=0.7697  val_loss=0.

In [44]:
# Final Test Evaluation

_, y_true, y_pred = run_epoch(test_loader, train=False)

print("\nTEST REPORT")
print(classification_report(
    y_true, y_pred,
    labels=list(label2id.values()),
    target_names=list(label2id.keys()),
    digits=3
))


TEST REPORT
              precision    recall  f1-score   support

          A1      0.867     0.845     0.856       193
          A2      0.654     0.812     0.724       144
          B1      0.930     0.831     0.878       160
          B2      0.829     0.746     0.785       130

    accuracy                          0.813       627
   macro avg      0.820     0.809     0.811       627
weighted avg      0.826     0.813     0.817       627

